### Chapter 5.4: 근육을 읽는 AI의 눈 - UNETR 근육 세그멘테이션
### 정형외과 수술의 정밀한 평가

[데이터셋 정보]
- 권장: TCIA (The Cancer Imaging Archive) CT 데이터
  - 다운로드: https://www.cancerimagingarchive.net/
  - 또는 CT-ORG 데이터셋 사용 가능
- 샘플 데이터: MSD (Medical Segmentation Decathlon) Task 10
  - 다운로드: http://medicaldecathlon.com/

[모델]
- UNETR (U-Net Transformer)
- Vision Transformer + U-Net 하이브리드
- 허벅지 근육 세그멘테이션 (4개 그룹)
  - Anterior (전면 근육)
  - Medial (내측 근육)
  - Gluteal (둔부 근육)
  - Posterior (후면 근육)

### Chapter 5.4: 근육 세그멘테이션 (UNETR + Total Segmentator)
#
### Total Segmentator의 마스크를 활용하여 허벅지 근육 4그룹 분할 모델 학습

##### 1. 라이브러리 설치


In [ ]:
# !pip install TotalSegmentator monai torch torchvision nibabel matplotlib SimpleITK scikit-image


In [ ]:
import os
import subprocess
import shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import nibabel as nib
import SimpleITK as sitk
from glob import glob

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from monai.networks.nets import UNETR
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd,
    Orientationd, ScaleIntensityRanged, CropForegroundd,
    RandCropByPosNegLabeld, ToTensord, MapTransform
)
from monai.data import CacheDataset

print("✅ 라이브러리 임포트 완료")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")

##### 2. Total Segmentator 설치 확인


In [ ]:
try:
    import totalsegmentator
    print("✅ TotalSegmentator 설치 확인")
    print(f"버전: {totalsegmentator.__version__}")
except ImportError:
    print("TotalSegmentator 설치 중...")
    !pip install TotalSegmentator
    print("✅ 설치 완료!")

##### 3. GPU 설정


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"사용 디바이스: {device}")

##### 4. 근육 클래스 및 매핑 정의


In [ ]:
# 목표: 4개 근육 그룹
MUSCLE_CLASSES = {
    0: 'Background',
    1: 'Anterior (전면 근육)',
    2: 'Medial (내측 근육)',
    3: 'Gluteal (둔부 근육)',
    4: 'Posterior (후면 근육)'
}

# Total Segmentator의 근육 클래스 → 4그룹 매핑
# Total Segmentator는 개별 근육을 분할하므로, 해부학적 위치에 따라 그룹화
TOTALSEG_TO_MUSCLE_MAP = {
    # Anterior (전면): 대퇴사두근, 장요근 등
    'iliopsoas_left': 1,
    'iliopsoas_right': 1,
    # 'quadriceps_left': 1,  # Total Segmentator에 없을 수 있음
    # 'quadriceps_right': 1,

    # Medial (내측): 내전근군
    # Total Segmentator의 기본 104개 클래스에는 개별 내전근이 없을 수 있음
    # 필요시 사용자 정의 매핑 추가

    # Gluteal (둔부): 대둔근, 중둔근, 소둔근
    'gluteus_maximus_left': 3,
    'gluteus_maximus_right': 3,
    'gluteus_medius_left': 3,
    'gluteus_medius_right': 3,
    'gluteus_minimus_left': 3,
    'gluteus_minimus_right': 3,

    # Posterior (후면): 햄스트링
    # 'hamstrings_left': 4,
    # 'hamstrings_right': 4,

    # 기타 하지 근육들
    'autochthon_left': 4,  # 고유 등근
    'autochthon_right': 4,
}

print("근육 클래스:")
for label, name in MUSCLE_CLASSES.items():
    print(f"  {label}: {name}")

print("\n✅ Total Segmentator 매핑 정의 완료")
print(f"매핑된 근육 수: {len(TOTALSEG_TO_MUSCLE_MAP)}")

##### 5. 데이터 경로 설정


In [ ]:
# 데이터 디렉토리 구조
DATA_ROOT = './data/muscle_training'
CT_INPUT_DIR = os.path.join(DATA_ROOT, 'ct_images')
TOTALSEG_OUTPUT_DIR = os.path.join(DATA_ROOT, 'totalseg_masks')
MAPPED_LABEL_DIR = os.path.join(DATA_ROOT, 'muscle_labels')

# 디렉토리 생성
os.makedirs(CT_INPUT_DIR, exist_ok=True)
os.makedirs(TOTALSEG_OUTPUT_DIR, exist_ok=True)
os.makedirs(MAPPED_LABEL_DIR, exist_ok=True)

print(f"데이터 루트: {DATA_ROOT}")
print(f"  - CT 입력: {CT_INPUT_DIR}")
print(f"  - Total Segmentator 출력: {TOTALSEG_OUTPUT_DIR}")
print(f"  - 매핑된 라벨: {MAPPED_LABEL_DIR}")

##### 6. Total Segmentator 실행 함수


In [ ]:
def run_totalsegmentator(input_ct_path, output_dir, fast=True):
    """
    Total Segmentator를 실행하여 근육 마스크 생성

    Args:
        input_ct_path: 입력 CT 파일 경로 (.nii.gz)
        output_dir: 출력 디렉토리
        fast: 빠른 모드 사용 여부
    """
    cmd = f'TotalSegmentator -i "{input_ct_path}" -o "{output_dir}"'

    if fast:
        cmd += ' --fast'

    print(f"🔬 Total Segmentator 실행 중...")
    print(f"명령어: {cmd}")

    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

    if result.returncode == 0:
        print("✅ Total Segmentator 완료!")
        return True
    else:
        print(f"❌ 실행 실패")
        print(f"에러: {result.stderr}")
        return False

In [ ]:
# 샘플 CT 다운로드 (테스트용)
import urllib.request

sample_url = "https://github.com/wasserth/TotalSegmentator/raw/master/resources/test_image.nii.gz"
sample_ct_path = os.path.join(CT_INPUT_DIR, 'sample_ct.nii.gz')

if not os.path.exists(sample_ct_path):
    print(f"샘플 CT 다운로드 중...")
    try:
        urllib.request.urlretrieve(sample_url, sample_ct_path)
        print(f"✅ 다운로드 완료: {sample_ct_path}")
    except Exception as e:
        print(f"❌ 다운로드 실패: {e}")
else:
    print(f"✅ 샘플 CT 존재: {sample_ct_path}")

##### 7. 근육 라벨 매핑 함수


In [ ]:
def map_totalseg_to_muscle_groups(totalseg_dir, output_path, mapping_dict):
    """
    Total Segmentator의 59개 클래스를 4개 근육 그룹으로 매핑

    Args:
        totalseg_dir: Total Segmentator 출력 디렉토리 (개별 .nii.gz 파일들)
        output_path: 매핑된 라벨 저장 경로
        mapping_dict: 클래스 매핑 딕셔너리

    Returns:
        매핑된 라벨 볼륨 (numpy array)
    """
    # Total Segmentator 출력 파일 목록
    mask_files = list(Path(totalseg_dir).glob('*.nii.gz'))

    if len(mask_files) == 0:
        print(f"⚠️  {totalseg_dir}에 마스크 파일이 없습니다.")
        return None

    # 첫 번째 파일에서 shape 가져오기
    first_mask = nib.load(mask_files[0])
    shape = first_mask.shape
    affine = first_mask.affine

    # 빈 라벨 볼륨 생성 (배경: 0)
    muscle_label = np.zeros(shape, dtype=np.uint8)

    print(f"근육 라벨 매핑 중... (총 {len(mask_files)}개 파일)")

    # 각 근육 마스크 처리
    for mask_file in mask_files:
        organ_name = mask_file.stem  # 파일명 (확장자 제외)

        # 매핑 딕셔너리에 있는 근육만 처리
        if organ_name in mapping_dict:
            target_class = mapping_dict[organ_name]

            # 마스크 로드
            mask_nii = nib.load(mask_file)
            mask_data = mask_nii.get_fdata()

            # 해당 근육 영역을 목표 클래스로 설정
            muscle_label[mask_data > 0] = target_class

            print(f"  ✓ {organ_name} → Class {target_class} ({MUSCLE_CLASSES[target_class]})")

    # NIfTI 파일로 저장
    muscle_label_nii = nib.Nifti1Image(muscle_label, affine)
    nib.save(muscle_label_nii, output_path)

    print(f"✅ 매핑 완료: {output_path}")

    # 통계 출력
    unique, counts = np.unique(muscle_label, return_counts=True)
    print("\n클래스 분포:")
    for cls, cnt in zip(unique, counts):
        if cls in MUSCLE_CLASSES:
            print(f"  Class {cls} ({MUSCLE_CLASSES[cls]}): {cnt:,} voxels")

    return muscle_label

##### 8. 전체 파이프라인 실행 (샘플 데이터)


In [ ]:
# Total Segmentator 실행
if os.path.exists(sample_ct_path):
    sample_output_dir = os.path.join(TOTALSEG_OUTPUT_DIR, 'sample_ct')

    if not os.path.exists(sample_output_dir) or len(os.listdir(sample_output_dir)) == 0:
        success = run_totalsegmentator(sample_ct_path, sample_output_dir, fast=True)
    else:
        print("✅ Total Segmentator 출력 이미 존재")
        success = True

    # 근육 라벨 매핑
    if success:
        mapped_label_path = os.path.join(MAPPED_LABEL_DIR, 'sample_ct_muscle_label.nii.gz')

        muscle_label = map_totalseg_to_muscle_groups(
            sample_output_dir,
            mapped_label_path,
            TOTALSEG_TO_MUSCLE_MAP
        )
else:
    print("⚠️  샘플 CT가 없습니다. 데이터를 준비하세요.")
    muscle_label = None


##### 9. Custom MONAI Transform - 근육 라벨 매핑


In [ ]:
class MapMuscleLabeld(MapTransform):
    """
    Total Segmentator 마스크를 4개 근육 그룹으로 매핑하는 MONAI Transform
    """
    def __init__(self, keys, mapping_dict):
        super().__init__(keys)
        self.mapping_dict = mapping_dict

    def __call__(self, data):
        d = dict(data)

        for key in self.keys:
            # 이미 매핑된 라벨이므로 그대로 사용
            # (또는 추가 처리가 필요한 경우 여기서 수행)
            pass

        return d

##### 10. MONAI Dataset 생성


In [ ]:
# 학습 데이터 리스트 생성
def prepare_data_dicts(ct_dir, label_dir):
    """
    MONAI Dataset을 위한 데이터 딕셔너리 생성
    """
    ct_files = sorted(glob(os.path.join(ct_dir, '*.nii.gz')))
    label_files = sorted(glob(os.path.join(label_dir, '*.nii.gz')))

    data_dicts = []
    for ct_file, label_file in zip(ct_files, label_files):
        data_dicts.append({
            'image': ct_file,
            'label': label_file
        })

    return data_dicts

In [ ]:
# 데이터 준비 (샘플)
train_files = prepare_data_dicts(CT_INPUT_DIR, MAPPED_LABEL_DIR)

if len(train_files) > 0:
    print(f"✅ 학습 데이터: {len(train_files)}개")
    for i, f in enumerate(train_files):
        print(f"  {i+1}. {Path(f['image']).name}")
else:
    print("⚠️  학습 데이터가 없습니다.")

##### 11. 데이터 전처리 Transform 정의


In [ ]:
# 학습용 transforms
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(keys=["image", "label"], pixdim=(1.5, 1.5, 1.5), mode=("bilinear", "nearest")),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    ScaleIntensityRanged(keys=["image"], a_min=-200, a_max=200, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(96, 96, 96),
        pos=1,
        neg=1,
        num_samples=4
    ),
    ToTensord(keys=["image", "label"])
])

print("✅ Transform 정의 완료")

In [ ]:
# Dataset 생성 (실제 데이터가 있을 때)
if len(train_files) > 0:
    train_ds = CacheDataset(
        data=train_files,
        transform=train_transforms,
        cache_rate=1.0,
        num_workers=4
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=2,
        shuffle=True,
        num_workers=0  # Windows에서는 0으로 설정
    )

    print(f"✅ DataLoader 생성 완료")
    print(f"  - 배치 크기: 2")
    print(f"  - 총 배치 수: {len(train_loader)}")
else:
    print("⚠️  데이터가 없어 DataLoader를 생성하지 않음")
    train_loader = None

##### 12. UNETR 모델 생성


In [ ]:
# UNETR 모델 초기화
model = UNETR(
    in_channels=1,
    out_channels=5,  # 배경 + 4개 근육 그룹
    img_size=(96, 96, 96),
    feature_size=16,
    hidden_size=768,
    mlp_dim=3072,
    num_heads=12,
    pos_embed='perceptron',
    norm_name='instance',
    res_block=True
)

model = model.to(device)

print("✅ UNETR 모델 생성 완료")

# 파라미터 수 확인
num_params = sum(p.numel() for p in model.parameters())
print(f"  - 총 파라미터 수: {num_params:,}")

##### 13. 손실 함수 및 최적화


In [ ]:
# Dice + Cross Entropy Loss
loss_function = DiceCELoss(to_onehot_y=True, softmax=True)

# AdamW optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Dice 메트릭
dice_metric = DiceMetric(include_background=False, reduction="mean")

print("✅ 손실 함수 및 최적화 설정 완료")

##### 14. 학습 루프


In [ ]:
# 실제 학습 (데이터가 있을 때 실행)
if train_loader is not None:
    NUM_EPOCHS = 50
    best_metric = -1

    print("\n학습 시작")
    print("="*60)

    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_loss = 0

        for batch_idx, batch_data in enumerate(train_loader):
            inputs = batch_data["image"].to(device)
            labels = batch_data["label"].to(device)

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = loss_function(outputs, labels)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{NUM_EPOCHS}], Loss: {avg_loss:.4f}")

        # 검증 (5 epoch마다)
        if (epoch + 1) % 5 == 0:
            model.eval()
            with torch.no_grad():
                for val_data in train_loader:  # 실제로는 별도의 val_loader 사용
                    val_inputs = val_data["image"].to(device)
                    val_labels = val_data["label"].to(device)

                    val_outputs = sliding_window_inference(
                        val_inputs, (96, 96, 96), 4, model
                    )

                    dice_metric(y_pred=val_outputs, y=val_labels)

            metric = dice_metric.aggregate().item()
            dice_metric.reset()

            print(f"  → 검증 Dice: {metric:.4f}")

            if metric > best_metric:
                best_metric = metric
                torch.save(model.state_dict(), './saved_models/best_unetr_muscle.pth')
                print(f"최고 성능 모델 저장! (Dice: {best_metric:.4f})")

    print("\n✅ 학습 완료!")
    print(f"최고 Dice 점수: {best_metric:.4f}")
else:
    print("학습 데이터가 준비되면 이 셀을 실행하세요.")

##### 15. 결과 시각화 (샘플)


In [ ]:
# 매핑된 라벨 시각화
if muscle_label is not None:
    # 원본 CT 로드
    ct_nii = nib.load(sample_ct_path)
    ct_volume = ct_nii.get_fdata()

    # 중간 슬라이스 선택
    middle_slice = ct_volume.shape[2] // 2

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # 원본 CT
    axes[0].imshow(ct_volume[:, :, middle_slice].T, cmap='gray', origin='lower')
    axes[0].set_title('원본 CT')
    axes[0].axis('off')

    # 매핑된 근육 라벨
    colors = ['black', 'red', 'blue', 'green', 'yellow']
    cmap = ListedColormap(colors)

    axes[1].imshow(muscle_label[:, :, middle_slice].T, cmap=cmap, origin='lower', vmin=0, vmax=4)
    axes[1].set_title('근육 라벨 (Total Segmentator 기반)')
    axes[1].axis('off')

    # 오버레이
    axes[2].imshow(ct_volume[:, :, middle_slice].T, cmap='gray', origin='lower')
    masked_seg = np.ma.masked_where(muscle_label[:, :, middle_slice].T == 0, muscle_label[:, :, middle_slice].T)
    axes[2].imshow(masked_seg, cmap=cmap, alpha=0.5, origin='lower', vmin=0, vmax=4)
    axes[2].set_title('오버레이')
    axes[2].axis('off')

    # 범례
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='red', label='Anterior'),
        Patch(facecolor='blue', label='Medial'),
        Patch(facecolor='green', label='Gluteal'),
        Patch(facecolor='yellow', label='Posterior')
    ]
    axes[2].legend(handles=legend_elements, loc='upper right', fontsize=8)

    plt.tight_layout()
    plt.savefig('muscle_label_totalseg.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("📊 시각화 저장: muscle_label_totalseg.png")

##### 16. 근육 부피 계산


In [ ]:
if muscle_label is not None:
    # 복셀 간격
    voxel_spacing = ct_nii.header.get_zooms()
    voxel_volume_cm3 = np.prod(voxel_spacing) / 1000.0  # mm³ → cm³

    print("\n근육 부피 분석")
    print("="*50)

    for label in [1, 2, 3, 4]:
        name = MUSCLE_CLASSES[label]
        num_voxels = np.sum(muscle_label == label)
        volume_cm3 = num_voxels * voxel_volume_cm3

        print(f"{name:25s}: {volume_cm3:8.2f} cm³")

    print("="*50)

##### 17. 배치 처리 함수


In [ ]:
def batch_process_ct_files(ct_dir, output_root):
    """
    여러 CT 파일을 배치로 처리

    Args:
        ct_dir: CT 파일들이 있는 디렉토리
        output_root: 출력 루트 디렉토리
    """
    ct_files = sorted(glob(os.path.join(ct_dir, '*.nii.gz')))

    print(f"배치 처리 시작: {len(ct_files)}개 파일")

    for i, ct_file in enumerate(ct_files, 1):
        ct_name = Path(ct_file).stem.replace('.nii', '')

        print(f"\n[{i}/{len(ct_files)}] 처리 중: {ct_name}")

        # Total Segmentator 출력 디렉토리
        totalseg_out = os.path.join(output_root, 'totalseg_masks', ct_name)

        # Total Segmentator 실행
        if not os.path.exists(totalseg_out):
            run_totalsegmentator(ct_file, totalseg_out, fast=True)
        else:
            print("  ✓ Total Segmentator 출력 이미 존재")

        # 근육 라벨 매핑
        mapped_label_path = os.path.join(output_root, 'muscle_labels', f'{ct_name}_muscle_label.nii.gz')

        if not os.path.exists(mapped_label_path):
            map_totalseg_to_muscle_groups(
                totalseg_out,
                mapped_label_path,
                TOTALSEG_TO_MUSCLE_MAP
            )
        else:
            print("  ✓ 근육 라벨 이미 존재")

    print("\n✅ 배치 처리 완료!")

In [ ]:
# 사용 예시 (주석 처리)
# batch_process_ct_files('./data/raw_ct_scans', DATA_ROOT)

##### 18. 임상 분석 요약


In [ ]:
print("\n" + "="*60)
print("📋 UNETR + Total Segmentator 파이프라인 요약")
print("="*60)

print("\n1. 데이터 준비:")
print("  ✓ Total Segmentator로 자동 세그멘테이션")
print("  ✓ 59개 클래스 → 4개 근육 그룹 매핑")
print("  ✓ 수동 라벨링 불필요")

print("\n2. 모델 정보:")
print("  - 아키텍처: UNETR (U-Net + Transformer)")
print(f"  - 파라미터 수: {num_params:,}")
print("  - 입력 크기: 96×96×96")
print("  - 출력 클래스: 5개 (배경 + 4개 근육)")

print("\n3. 학습 데이터:")
print("  - 소스: Total Segmentator 마스크")
print("  - 장점: 대량 데이터 확보 용이")
print("  - 품질: 일관된 자동 라벨링")

print("\n4. 임상 활용:")
print("  - 정형외과 수술 전후 평가")
print("  - 근육 위축/손상 모니터링")
print("  - 재활 치료 효과 평가")

print("="*60)